# Lab Day 10 — Qwen3-VL-4B, QLoRA và định vị hư hại mặt đường
Triển khai đầy đủ E0–E4. Kết quả được chấm bằng công cụ gốc; không điền số liệu trước khi chạy. Mở runtime T4. Mỗi run khởi tạo lại model gốc; adapter và projector lưu ngoài Git.


## 0. Môi trường (Colab T4)
Notebook ghi mã thực hiện vào runtime nên có thể chạy lại độc lập. Test dành riêng cho REPORT còn placeholder được bỏ qua khi kiểm tra repo đã điền báo cáo; giữ nguyên file test.

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = Path('/content/road_damage_lab')
if not (REPO / 'README.md').exists():
    subprocess.run(['git', 'clone', 'https://github.com/nguyenvantu19/K4-Track-4-Day10-VisionLanguageModel.git', str(REPO)], check=True)
os.chdir(REPO)
!nvidia-smi -L
!pip -q install -r requirements-colab.txt pytest
# torchao không dùng trong lab bitsandbytes; bản có sẵn ở Colab xung đột PEFT.
!pip -q uninstall -y torchao
!python -m pytest -q -k 'not test_report_template_has_student_markers_and_ten_questions'


## 1. Dữ liệu: tải và chia split theo phiên
Dùng ZIP công khai của dataset đúng đề. Công cụ prepare_data giữ nguyên, không tự chia lại split.

In [ ]:
if not Path('data/splits/meta.json').exists():
    subprocess.run([sys.executable, 'scripts/prepare_data.py', '--zip', 'https://www.kaggle.com/api/v1/datasets/download/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes'], check=True)
import json
meta = json.loads(Path('data/splits/meta.json').read_text())
print(meta['dataset_version'])
print(json.dumps(meta['counts'], indent=2))


## 2. Nhãn YOLO → JSON thang 0–1000
Cell sau chứa mã triển khai: chuyển nhãn, encode và mask, LoRA, train, lưu/nạp projector, inference và kiểm tra kiến trúc. `yolo_to_box` và `objects_to_answer` được dùng thống nhất cho dữ liệu và ảnh kiểm chứng.

In [ ]:
%%writefile student_lab.py
"""Student implementation for Day 10. Instructor scripts/tests stay unchanged.

Run from repository root on a Colab CUDA runtime. CLI examples are in the
completed lab_day10.ipynb. Results are always generated by scripts/score.py.
"""
from __future__ import annotations

import argparse
import functools
import gc
import json
import math
import random
import re
import subprocess
import sys
import time
from pathlib import Path

from scripts.labcommon import CLASSES, read_jsonl

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
PROMPT = (
    "Locate every pothole, crack, and manhole in this road image. "
    "Return ONLY a JSON array, with no explanation or markdown. "
    'Each object must have "label" and "bbox_2d": [x1,y1,x2,y2]. '
    "Allowed labels are pothole, crack, manhole. Use integer coordinates "
    "normalized to 0-1000 relative to image width and height. "
    "Return [] if there are no objects."
)
PROJECTORS = ("visual.merger", "visual.deepstack_merger_list")
LLM_PATTERN = r"model\.language_model\.layers\.\d+\.(?:self_attn\.(?:q|k|v|o)_proj|mlp\.(?:gate|up|down)_proj)"
VIT_PATTERN = r"model\.visual\.blocks\.\d+\.(?:attn\.(?:qkv|proj)|mlp\.(?:linear_fc1|linear_fc2))"


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def yolo_to_box(label, yolo):
    xc, yc, w, h = yolo
    values = (xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2)
    return {"label": label, "bbox_2d": [max(0, min(1000, int(round(v * 1000)))) for v in values]}


def objects_to_answer(objects):
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda o: (o["bbox_2d"][1], o["bbox_2d"][0], o["label"]))
    return json.dumps(boxes, separators=(",", ":"))


def seed_everything(seed):
    import numpy as np
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_processor():
    from transformers import AutoProcessor
    return AutoProcessor.from_pretrained(MODEL_ID)


def load_base():
    import torch
    from transformers import BitsAndBytesConfig, Qwen3VLForConditionalGeneration
    from transformers.utils import logging
    logging.disable_progress_bar()
    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required. Select Colab T4 before running.")
    quant = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        # Transformers 5 matches prefixes, not arbitrary path segments.
        llm_int8_skip_modules=["model.visual", "visual", "lm_head"],
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quant, dtype=torch.float16,
        device_map={"": 0}, attn_implementation="sdpa",
    )
    from bitsandbytes.nn import Linear4bit
    wrongly_quantized = [name for name, module in model.named_modules()
                         if "visual" in name and isinstance(module, Linear4bit)]
    if wrongly_quantized:
        raise RuntimeError(f"Visual modules must remain float16: {wrongly_quantized[:3]}")
    return model


def user_message(row):
    return {"role": "user", "content": [
        {"type": "image", "image": str((Path("data") / row["image_path"]).resolve())},
        {"type": "text", "text": PROMPT},
    ]}


def encode_example(processor, row):
    import torch
    user = user_message(row)
    common = dict(tokenize=True, return_dict=True, return_tensors="pt")
    prefix = processor.apply_chat_template([user], add_generation_prompt=True, **common)
    full = processor.apply_chat_template(
        [user, {"role": "assistant", "content": objects_to_answer(row["objects"])}],
        add_generation_prompt=False, **common,
    )
    n = prefix["input_ids"].shape[1]
    if not torch.equal(prefix["input_ids"], full["input_ids"][:, :n]):
        raise ValueError(f"Chat template prefix mismatch: {row['image_id']}")
    full["labels"] = full["input_ids"].clone()
    full["labels"][:, :n] = -100
    full["labels"][full["attention_mask"] == 0] = -100
    if not (full["labels"] != -100).any():
        raise ValueError("Example has no answer tokens")
    return dict(full)


def collate(batch, pad_id):
    import torch
    from torch.nn.utils.rnn import pad_sequence
    result = {}
    for key in ("input_ids", "attention_mask", "mm_token_type_ids", "labels"):
        if key not in batch[0]:
            continue
        padding = -100 if key == "labels" else pad_id if key == "input_ids" else 0
        result[key] = pad_sequence([b[key].squeeze(0) for b in batch], batch_first=True, padding_value=padding)
    for key in ("pixel_values", "image_grid_thw"):
        if key in batch[0]:
            result[key] = torch.cat([b[key] for b in batch], dim=0)
    return result


def to_device(batch):
    import torch
    return {key: value.to("cuda", dtype=torch.float16 if key == "pixel_values" else value.dtype)
            for key, value in batch.items()}


def apply_setup(base, setup, rank=16):
    import torch
    from peft import LoraConfig, get_peft_model
    for param in base.parameters():
        param.requires_grad_(False)
    base.enable_input_require_grads()
    pattern = LLM_PATTERN if setup != "E3" else f"(?:{LLM_PATTERN}|{VIT_PATTERN})"
    matched = [name for name, module in base.named_modules() if re.fullmatch(pattern, name)]
    expected = 348 if setup == "E3" else 252
    if len(matched) != expected:
        raise ValueError(f"Expected {expected} LoRA targets, found {len(matched)}. Inspect named_modules.")
    model = get_peft_model(base, LoraConfig(
        r=rank, lora_alpha=2 * rank, lora_dropout=0.05,
        target_modules=matched, bias="none", task_type="CAUSAL_LM",
    ))
    if setup in ("E2", "E3"):
        for name, param in model.named_parameters():
            if any(part in name for part in PROJECTORS):
                if param.__class__.__name__ == "Params4bit":
                    raise TypeError(f"Cannot fully train quantized projector: {name}")
                param.data = param.data.to(torch.float32)
                param.requires_grad_(True)
    # ViT LoRA can inherit fp16 from unquantized visual weights. GradScaler
    # requires fp32 trainable weights/gradients; autocast still computes fp16.
    for param in model.parameters():
        if param.requires_grad:
            param.data = param.data.to(torch.float32)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{setup}: {len(matched)} LoRA modules, {trainable:,} trainable parameters", flush=True)
    return model, matched, trainable


def save_checkpoint(model, directory, info, optimizer=None, scheduler=None, scaler=None):
    import torch
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(directory / "adapter", safe_serialization=True)
    extra = {name: p.detach().cpu() for name, p in model.named_parameters()
             if p.requires_grad and any(part in name for part in PROJECTORS)}
    if extra:
        torch.save(extra, directory / "projector.pt")
    if optimizer is not None:
        torch.save({"optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                    "scaler": scaler.state_dict(), "step": info["completed_steps"],
                    "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state(),
                    "python_rng": random.getstate()}, directory / "training_state.pt")
    write_json(directory / "run.json", info)
    print("CHECKPOINT_READY " + json.dumps({"directory": str(directory),
          "name": info["name"], "step": info["completed_steps"]}), flush=True)


def restore_projector(model, path):
    import torch
    path = Path(path)
    if not path.exists():
        return
    state = torch.load(path, map_location="cpu", weights_only=True)
    named = dict(model.named_parameters())
    unknown = set(state) - set(named)
    if unknown:
        raise ValueError(f"Projector parameter mismatch: {sorted(unknown)[:3]}")
    for name, value in state.items():
        named[name].data = value.to(device=named[name].device, dtype=value.dtype)
    print(f"Restored {len(state)} projector tensors", flush=True)


def train(args):
    import torch
    from torch.utils.data import DataLoader, Dataset
    from transformers import get_cosine_schedule_with_warmup
    seed_everything(args.seed)
    records = read_jsonl("data/splits/train.jsonl")
    if not 0 < args.fraction <= 1:
        raise ValueError("fraction must be in (0,1]")
    if args.fraction < 1:
        chosen = random.Random(args.seed).sample(range(len(records)), max(1, int(len(records) * args.fraction)))
        records = [records[i] for i in sorted(chosen)]
    processor = get_processor()
    processor.tokenizer.padding_side = "right"
    class RoadDataset(Dataset):
        def __len__(self):
            return len(records)
        @functools.lru_cache(maxsize=32)
        def __getitem__(self, i):
            return encode_example(processor, records[i])
    dataset = RoadDataset()
    sample = dataset[0]
    answer = processor.tokenizer.decode(sample["labels"][sample["labels"] != -100])
    image_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
    n_image = int((sample["input_ids"] == image_id).sum())
    print("Mask verification:", answer, "image tokens:", n_image, flush=True)
    if n_image != 220:
        raise ValueError(f"Expected 220 image tokens for this lab, got {n_image}")
    base = load_base()
    model, matched, trainable = apply_setup(base, args.setup, args.rank)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    model.train()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=args.lr, weight_decay=0, foreach=False)
    scheduler = get_cosine_schedule_with_warmup(optimizer, math.ceil(args.steps * 0.05), args.steps)
    scaler = torch.amp.GradScaler("cuda", init_scale=1024)
    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, num_workers=0,
                        collate_fn=functools.partial(collate, pad_id=processor.tokenizer.pad_token_id))
    iterator = iter(loader)
    directory = Path(args.output)
    directory.mkdir(parents=True, exist_ok=True)
    info = {"name": args.name, "setup": args.setup, "model_id": MODEL_ID,
            "trainable_params": trainable, "matched_modules": matched,
            "train_images": len(records), "train_fraction": args.fraction,
            "rank": args.rank, "alpha": 2 * args.rank, "seed": args.seed,
            "batch_size": args.batch_size, "grad_accum": args.grad_accum,
            "effective_batch_size": args.batch_size * args.grad_accum,
            "max_steps": args.steps, "learning_rate": args.lr,
            "warmup_steps": math.ceil(args.steps * 0.05), "mask_answer": answer,
            "image_tokens": n_image, "completed_steps": 0,
            "dataset_version": json.loads(Path("data/splits/meta.json").read_text())["dataset_version"]}
    start_step = 0
    previous_minutes = previous_peak = 0
    if args.resume:
        resume = Path(args.resume)
        previous = json.loads((resume / "run.json").read_text())
        for key in ("setup", "rank", "seed", "train_fraction", "max_steps", "batch_size", "grad_accum", "dataset_version"):
            if info[key] != previous[key]:
                raise ValueError(f"Resume config mismatch: {key}")
        from peft import set_peft_model_state_dict
        from safetensors.torch import load_file
        set_peft_model_state_dict(model, load_file(str(resume / "adapter" / "adapter_model.safetensors")))
        restore_projector(model, resume / "projector.pt")
        state = torch.load(resume / "training_state.pt", map_location="cpu", weights_only=False)
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        start_step = state["step"]
        torch.set_rng_state(state["torch_rng"])
        torch.cuda.set_rng_state(state["cuda_rng"])
        random.setstate(state["python_rng"])
        previous_minutes = previous["train_minutes"]
        previous_peak = previous["peak_vram_gb"]
        info["resume_note"] = "Optimizer/RNG restored; DataLoader restarts at a shuffled epoch."
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()
    optimizer.zero_grad(set_to_none=True)
    log_path = directory / "train_log.jsonl"
    with log_path.open("a" if args.resume else "w", encoding="utf-8") as log:
        step = start_step + 1
        consecutive_overflows = 0
        while step <= args.steps:
            total_loss = 0.0
            for _ in range(args.grad_accum):
                try:
                    batch = next(iterator)
                except StopIteration:
                    iterator = iter(loader)
                    batch = next(iterator)
                batch = to_device(batch)
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = model(**batch).loss
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Nonfinite loss at optimizer step {step}")
                total_loss += float(loss.detach()) / args.grad_accum
                scaler.scale(loss / args.grad_accum).backward()
                del loss, batch
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
            if not torch.isfinite(grad_norm):
                # GradScaler records overflow during unscale_ and skips this update.
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                consecutive_overflows += 1
                print(f"Gradient overflow: retry step {step}, scale={scaler.get_scale()}", flush=True)
                if consecutive_overflows >= 10:
                    raise FloatingPointError("Repeated gradient overflow; reduce learning rate and inspect inputs")
                continue
            consecutive_overflows = 0
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            elapsed = previous_minutes + (time.perf_counter() - started) / 60
            row = {"step": step, "loss": total_loss, "lr": scheduler.get_last_lr()[0],
                   "minutes": elapsed, "grad_norm": float(grad_norm)}
            log.write(json.dumps(row) + "\n")
            log.flush()
            if step == 1 or step % 10 == 0:
                print(f"{args.name} step {step}/{args.steps}: loss={total_loss:.4f}, {elapsed:.1f} min", flush=True)
            info.update(completed_steps=step, train_minutes=elapsed,
                        peak_vram_gb=max(previous_peak, torch.cuda.max_memory_allocated() / 1024 ** 3))
            write_json(directory / "progress.json", info)
            if step % 100 == 0 or step == args.steps:
                save_checkpoint(model, directory, info, optimizer, scheduler, scaler)
            step += 1
    processor.save_pretrained(directory / "processor")
    print(f"Finished {args.name}: {info['train_minutes']:.2f} min, {info['peak_vram_gb']:.2f} GiB", flush=True)


def infer(args):
    import torch
    from peft import PeftModel
    seed_everything(args.seed)
    records = read_jsonl(f"data/splits/{args.split}.jsonl")
    if args.limit:
        records = records[:args.limit]
    processor = get_processor()
    processor.tokenizer.padding_side = "left"
    model = load_base()
    if args.checkpoint:
        checkpoint = Path(args.checkpoint)
        run = json.loads((checkpoint / "run.json").read_text())
        if run["completed_steps"] != run["max_steps"]:
            raise ValueError("Refusing to evaluate an incomplete training run")
        model = PeftModel.from_pretrained(model, checkpoint / "adapter", is_trainable=False)
        if run["setup"] in ("E2", "E3") and not (checkpoint / "projector.pt").exists():
            raise FileNotFoundError("E2/E3 checkpoint is missing trained projector")
        restore_projector(model, checkpoint / "projector.pt")
    model.eval()
    model.config.use_cache = True
    model.generation_config.do_sample = False
    out_path = Path(args.output)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    already = read_jsonl(out_path) if args.resume and out_path.exists() else []
    seen = {r["image_id"] for r in already}
    records = [r for r in records if r["image_id"] not in seen]
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    with out_path.open("a" if args.resume else "w", encoding="utf-8") as stream:
        for offset in range(0, len(records), args.batch_size):
            rows = records[offset:offset + args.batch_size]
            conversations = [[user_message(row)] for row in rows]
            inputs = processor.apply_chat_template(
                conversations, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors="pt", padding=True,
            )
            inputs = to_device(dict(inputs))
            with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
                generated = model.generate(**inputs, max_new_tokens=args.max_new_tokens, do_sample=False)
            outputs = processor.batch_decode(generated[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for row, raw in zip(rows, outputs):
                stream.write(json.dumps({"image_id": row["image_id"], "raw": raw}, ensure_ascii=False) + "\n")
            stream.flush()
            print(f"{args.name} {args.split}: {len(seen) + offset + len(rows)} outputs, {(time.perf_counter()-started)/60:.1f} min", flush=True)
            del generated, inputs
    write_json(out_path.with_suffix(".run.json"), {
        "name": args.name, "split": args.split, "checkpoint": args.checkpoint,
        "max_new_tokens": args.max_new_tokens, "batch_size": args.batch_size,
        "inference_minutes": (time.perf_counter() - started) / 60,
        "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024 ** 3,
    })


def inspect_data(args):
    from PIL import Image, ImageDraw
    meta = json.loads(Path("data/splits/meta.json").read_text())
    print(json.dumps(meta["counts"], indent=2))
    splits = {s: read_jsonl(f"data/splits/{s}.jsonl") for s in ("train", "val", "test")}
    sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        assert sessions[a].isdisjoint(sessions[b]), f"Session leakage: {a}/{b}"
    folder = Path("report/figures")
    folder.mkdir(parents=True, exist_ok=True)
    colors = {"pothole": "red", "crack": "orange", "manhole": "cyan"}
    for i, row in enumerate(splits["train"][:6]):
        image = Image.open(Path("data") / row["image_path"]).convert("RGB")
        draw = ImageDraw.Draw(image)
        for obj in json.loads(objects_to_answer(row["objects"])):
            x1, y1, x2, y2 = obj["bbox_2d"]
            box = (x1/1000*image.width, y1/1000*image.height, x2/1000*image.width, y2/1000*image.height)
            draw.rectangle(box, outline=colors[obj["label"]], width=2)
            draw.text((box[0], max(0, box[1]-12)), obj["label"], fill=colors[obj["label"]])
        image.save(folder / f"ground_truth_{i+1}.jpg")
    assert yolo_to_box("pothole", [0.5, 0.5, 0.2, 0.2])["bbox_2d"] == [400, 400, 600, 600]
    assert objects_to_answer([]) == "[]"
    write_json("report/data_checks.json", {"dataset_version": meta["dataset_version"],
        "counts": meta["counts"], "session_overlap": False, "yolo_conversion_verified": True})
    print("Session and coordinate checks passed; 6 label figures saved.")


def verify_model(args):
    import torch
    from accelerate import init_empty_weights
    from transformers import AutoConfig, Qwen3VLForConditionalGeneration
    processor = get_processor()
    row = read_jsonl("data/splits/train.jsonl")[0]
    encoded = encode_example(processor, row)
    print("Answer-only labels:", processor.tokenizer.decode(encoded["labels"][encoded["labels"] != -100]))
    image_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
    count = int((encoded["input_ids"] == image_id).sum())
    print("Image tokens:", count)
    assert count == 220
    with init_empty_weights():
        empty = Qwen3VLForConditionalGeneration(AutoConfig.from_pretrained(MODEL_ID))
    # Accelerate initialization may break shared embedding/head identity.
    # Tie outside the context before counting unique logical parameters.
    empty.tie_weights()
    groups = {"llm": 0, "vit_backbone": 0, "projector": 0, "total": 0}
    for name, param in empty.named_parameters():
        n = param.numel()
        groups["total"] += n
        key = "projector" if any(part in name for part in PROJECTORS) else "vit_backbone" if "visual" in name else "llm"
        groups[key] += n
    names = [name for name, _ in empty.named_modules()]
    targets = {"llm": [n for n in names if re.fullmatch(LLM_PATTERN, n)],
               "vit": [n for n in names if re.fullmatch(VIT_PATTERN, n)]}
    assert len(targets["llm"]) == 252 and len(targets["vit"]) == 96
    write_json("report/model_checks.json", {"parameters": groups, "targets": targets,
        "image_tokens": count, "decoded_labels": processor.tokenizer.decode(encoded["labels"][encoded["labels"] != -100])})
    print(json.dumps(groups, indent=2))
    del empty
    gc.collect()


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    sub = parser.add_subparsers(dest="command", required=True)
    sub.add_parser("inspect-data")
    sub.add_parser("verify-model")
    p = sub.add_parser("train")
    p.add_argument("--setup", choices=("E1", "E2", "E3"), required=True)
    p.add_argument("--name", required=True)
    p.add_argument("--output", required=True)
    p.add_argument("--rank", type=int, default=16)
    p.add_argument("--fraction", type=float, default=1.0)
    p.add_argument("--steps", type=int, default=300)
    p.add_argument("--batch-size", type=int, default=2)
    p.add_argument("--grad-accum", type=int, default=4)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--resume")
    p = sub.add_parser("infer")
    p.add_argument("--name", required=True)
    p.add_argument("--output", required=True)
    p.add_argument("--checkpoint")
    p.add_argument("--split", choices=("val", "test"), default="test")
    p.add_argument("--batch-size", type=int, default=4)
    p.add_argument("--max-new-tokens", type=int, default=256)
    p.add_argument("--limit", type=int)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--resume", action="store_true")
    args = parser.parse_args()
    {"inspect-data": inspect_data, "verify-model": verify_model, "train": train, "infer": infer}[args.command](args)


if __name__ == "__main__":
    main()


In [ ]:
!python student_lab.py inspect-data
from IPython.display import display
from PIL import Image
display(Image.open('report/figures/ground_truth_1.jpg'))


## 3. E0: zero-shot
Model NF4, bỏ lượng tử hóa visual/lm_head, padding trái, decode riêng phần sinh thêm. Pipeline ở bước 8 chạy đủ test và chấm E0. Có thể dùng cell này để kiểm tra hai ảnh val trước.

In [ ]:
!python -u student_lab.py infer --name E0_debug --split val --limit 2 --batch-size 2 --output work/e0_debug.jsonl
print(Path('work/e0_debug.jsonl').read_text())


## 4. Chat template và mask loss
Kiểm tra prompt là tiền tố của full input. Labels prompt/padding=-100; pixel_values và image_grid_thw được nối, không pad. Decode phần labels không mask để kiểm tra JSON và token kết thúc.

In [ ]:
!python student_lab.py verify-model
checks = json.loads(Path('report/model_checks.json').read_text())
print('Image tokens:', checks['image_tokens'])
print('Answer-only labels:', checks['decoded_labels'])
print('Parameter counts:', checks['parameters'])


## 5. Setup LoRA E1/E2/E3
E1: LoRA LLM; E2: thêm projector toàn phần float32; E3: thêm LoRA ViT. Target module được kiểm tra bằng tên đầy đủ: 252 LLM, 96 ViT. Rank 16, alpha 32, dropout 0.05.

In [ ]:
print('LLM targets:', len(checks['targets']['llm']))
print('ViT targets:', len(checks['targets']['vit']))
assert len(checks['targets']['llm']) == 252
assert len(checks['targets']['vit']) == 96


## 6. Vòng huấn luyện và checkpoint
AdamW, warmup 5%, cosine, 300 optimizer steps, gradient checkpointing, fp16 GradScaler và clip norm 1. Mỗi 100 steps lưu adapter, projector riêng, optimizer/scheduler/scaler và metadata. Overflow gradient sẽ giảm loss scale và thử lại, không được tính thành optimizer step. Có thể lưu checkpoint vào Drive nếu Drive đã được mount; mặc định dùng work/checkpoints trong runtime.

In [ ]:
%%writefile make_report.py
"""Build the lab report and annotated figures from measured results only."""
import argparse
import json
import statistics
import subprocess
import sys
from pathlib import Path

from scripts.labcommon import CLASSES, read_jsonl
from scripts.score import _gt_boxes, _match, evaluate, parse_output


def load(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def fnum(value):
    return f"{value:.4f}"


def ci(result):
    return "[" + ", ".join(fnum(x) for x in result["f1_ci95"]) + "]"


def generate(name):
    from PIL import Image, ImageDraw
    meta = load("data/splits/meta.json")
    model = load("report/model_checks.json")
    evidence = {key: load(f"report/evidence/{key}.json") for key in ("e0", "e1", "e2", "e3")}
    # Earlier pipeline revisions scored E0 without resource metadata. Add only
    # measured metadata through the original scorer, retaining identical outputs.
    if "peak_vram_gb" not in evidence["e0"].get("meta", {}):
        e0_run = load("work/e0.run.json")
        subprocess.run([sys.executable, "scripts/score.py", "--pred", "work/e0.jsonl",
                        "--name", "E0", "--out", "report/evidence/e0.json",
                        "--meta", "trainable_params=0", "--meta", "train_minutes=0",
                        "--meta", f"peak_vram_gb={e0_run['peak_vram_gb']}"], check=True)
        evidence["e0"] = load("report/evidence/e0.json")
    e4_paths = sorted(Path("report/evidence").glob("e4*.json"))
    if len(e4_paths) != 2:
        raise ValueError("Both E4 data-size conditions must be scored before report generation")
    evidence.update({p.stem: load(p) for p in e4_paths})
    entries = read_jsonl("data/splits/test.jsonl")
    best = load("report/selection.json")["best_setup"]
    pred_rows = read_jsonl(f"work/{best.lower()}.jsonl")
    preds = {r["image_id"]: r["raw"] for r in pred_rows}
    folder = Path("report/figures")
    folder.mkdir(parents=True, exist_ok=True)
    candidates = []
    for entry in entries:
        parsed = parse_output(preds[entry["image_id"]])
        per, _ = _match(_gt_boxes(entry), parsed or [])
        counts = [sum(v[i] for v in per.values()) for i in range(3)]
        candidates.append((entry, parsed, counts))
    selected = []
    used = set()
    # Include success and failures, plus class-specific examples when available.
    groups = [lambda x: x[2][0] > 0 and x[2][1] == x[2][2] == 0,
              lambda x: x[2][1] > 0, lambda x: x[2][2] > 0]
    groups += [(lambda x, lab=lab: any(o["label"] == lab for o in x[0]["objects"])) for lab in CLASSES]
    for condition in groups:
        for candidate in candidates:
            if candidate[0]["image_id"] not in used and condition(candidate):
                selected.append(candidate)
                used.add(candidate[0]["image_id"])
                break
    for candidate in candidates:
        if len(selected) >= 6:
            break
        if candidate[0]["image_id"] not in used:
            selected.append(candidate)
            used.add(candidate[0]["image_id"])
    illustrations = []
    for i, (entry, parsed, counts) in enumerate(selected[:6], 1):
        image = Image.open(Path("data") / entry["image_path"]).convert("RGB")
        draw = ImageDraw.Draw(image)
        for boxes, color, prefix in ((_gt_boxes(entry), "lime", "GT"), (parsed or [], "red", "PRED")):
            for box in boxes:
                x1, y1, x2, y2 = box["bbox"]
                xy = (x1/1000*image.width, y1/1000*image.height, x2/1000*image.width, y2/1000*image.height)
                draw.rectangle(xy, outline=color, width=2)
                draw.text((max(0, xy[0]), max(0, xy[1]-12)), prefix+":"+box["label"], fill=color)
        draw.rectangle((0, 0, image.width, 18), fill="black")
        draw.text((4, 3), f"{best} | TP={counts[0]} FP={counts[1]} FN={counts[2]} | GT green / PRED red", fill="white")
        path = folder / f"prediction_{i}.jpg"
        image.save(path, quality=92)
        illustrations.append(f"![Ví dụ {i}]({path.as_posix()})\n\n"
            f"Ảnh `{entry['image_id']}`: TP={counts[0]}, FP={counts[1]}, FN={counts[2]}; "
            f"JSON {'hợp lệ' if parsed is not None else 'không hợp lệ'}. Box thật màu xanh, dự đoán màu đỏ.")
    area = {lab: [] for lab in CLASSES}
    for entry in entries:
        for obj in entry["objects"]:
            area[obj["label"]].append(obj["yolo"][2] * obj["yolo"][3] * 100)
    area_text = "; ".join(f"{lab}: {statistics.median(area[lab]):.3f}%" for lab in CLASSES)
    e0_raw = {r["image_id"]: r["raw"] for r in read_jsonl("work/e0.jsonl")}
    error_summary = {"invalid_json": 0, "images_with_fp": 0, "images_with_fn": 0,
                     "predictions_unknown_label": 0, "boxes_outside_0_1000": 0}
    examples = []
    for entry in entries:
        raw = e0_raw[entry["image_id"]]
        parsed = parse_output(raw)
        if parsed is None:
            error_summary["invalid_json"] += 1
        else:
            error_summary["predictions_unknown_label"] += sum(p["label"] not in CLASSES for p in parsed)
            error_summary["boxes_outside_0_1000"] += sum(any(v < 0 or v > 1000 for v in p["bbox"]) for p in parsed)
        per, _ = _match(_gt_boxes(entry), parsed or [])
        fp, fn = sum(v[1] for v in per.values()), sum(v[2] for v in per.values())
        error_summary["images_with_fp"] += fp > 0
        error_summary["images_with_fn"] += fn > 0
        if (fp or fn or parsed is None) and len(examples) < 2:
            examples.append(f"Ảnh `{entry['image_id']}`, FP={fp}, FN={fn}:\n\n```text\n{raw}\n```")
    Path("report/e0_error_analysis.json").write_text(json.dumps(error_summary, indent=2), encoding="utf-8")
    category_names = {"invalid_json": "JSON không parse được", "images_with_fp": "phát hiện thừa/sai (FP)",
                      "images_with_fn": "bỏ sót (FN)"}
    dominant = sorted(category_names, key=lambda key: error_summary[key], reverse=True)[:2]
    dominant_text = " và ".join(f"{category_names[key]} ({error_summary[key]} ảnh)" for key in dominant)
    e1_preds = {r["image_id"]: r["raw"] for r in read_jsonl("work/e1.jsonl")}
    wrong = evaluate(entries, e1_preds, coord="pixel")
    Path("report/coordinate_diagnostic.json").write_text(json.dumps(wrong, indent=2), encoding="utf-8")
    table = ["| Mã | Setup | Tham số huấn luyện | VRAM đỉnh (GiB) | Phút train | F1 | CI95 | FP/ảnh | F1 pothole | F1 crack | F1 manhole |",
             "|---|---|---:|---:|---:|---:|---|---:|---:|---:|---:|"]
    setup_labels = {"e0": "Zero-shot", "e1": "LoRA LLM", "e2": "+ projector", "e3": "+ LoRA ViT"}
    for key, result in evidence.items():
        m = result.get("meta", {})
        description = setup_labels.get(key, key.replace("e4_data_size_", "E4 train_fraction="))
        table.append(f"| {result['name']} | {description} | {int(m.get('trainable_params', 0)):,} | "
            f"{m.get('peak_vram_gb',0):.3f} | {m.get('train_minutes',0):.2f} | {fnum(result['micro']['f1'])} | "
            f"{ci(result)} | {result['fp_per_image']:.3f} | " + " | ".join(fnum(result["per_class"][lab]["f1"]) for lab in CLASSES) + " |")
    pp = model["parameters"]
    comparison = "\n".join(f"- {key.upper()}: F1={fnum(evidence[key]['micro']['f1'])}, CI95={ci(evidence[key])}, "
        f"VRAM={evidence[key]['meta']['peak_vram_gb']:.3f} GiB, train={evidence[key]['meta']['train_minutes']:.2f} phút."
        for key in ("e1", "e2", "e3"))
    counts = "\n".join(f"- {s}: {meta['counts'][s]['images']} ảnh; " + ", ".join(f"{c}={meta['counts'][s]['boxes'][c]} box" for c in CLASSES) for s in ("train", "val", "test"))
    worst = min(CLASSES, key=lambda c: evidence[best.lower()]["per_class"][c]["f1"])
    peak_key = max(("e1", "e2", "e3"), key=lambda k: evidence[k]["meta"]["peak_vram_gb"])
    delta_params = evidence["e2"]["meta"]["trainable_params"] - evidence["e1"]["meta"]["trainable_params"]
    e4_description = "\n".join(f"- {k}: F1={fnum(evidence[k]['micro']['f1'])}, CI95={ci(evidence[k])}." for k in evidence if k.startswith("e4"))
    widths = "; ".join(f"{k.upper()}: {evidence[k]['f1_ci95'][1]-evidence[k]['f1_ci95'][0]:.4f}" for k in ("e0", "e1", "e2", "e3"))
    selection = load("report/selection.json")
    report = f'''Họ tên: {name}
`dataset_version`: `{meta['dataset_version']}`

## Bảng tổng hợp

{chr(10).join(table)}

Các kết quả do `scripts/score.py` gốc tạo trên toàn bộ test; không sửa công cụ chấm. VRAM dùng `max_memory_allocated()/1024**3`, đơn vị GiB. Thời gian ở bảng là thời gian vòng train, gồm các checkpoint trước bước cuối, không gồm nạp model hay lưu checkpoint cuối; E0 không train. Thời gian suy luận được lưu riêng trong `work/*.run.json`.

## 10 câu hỏi

### Câu 1 — Kiến trúc và các khối được cập nhật

Đếm trên model chưa lượng tử hóa, khởi tạo ở meta device: LLM và phần embedding/head ngoài visual có {pp['llm']:,} tham số; backbone ViT có {pp['vit_backbone']:,}; projector có {pp['projector']:,}; tổng {pp['total']:,}. ViT trích đặc trưng ảnh; projector và các merger DeepStack đưa đặc trưng về không gian của LLM; LLM sinh JSON. E1 chỉ cập nhật LoRA LLM, E2 cập nhật thêm toàn bộ projector, E3 thêm LoRA backbone ViT. Số target đã xác nhận: {len(model['targets']['llm'])} LLM và {len(model['targets']['vit'])} ViT. E2 thêm {delta_params:,} tham số trainable so với E1.

### Câu 2 — Output và lỗi của E0

E0 có parse_valid_rate={evidence['e0']['parse_valid_rate']:.4f}. Kiểm tra từng ảnh ghi nhận {error_summary['invalid_json']} output không parse được, {error_summary['images_with_fp']} ảnh có FP và {error_summary['images_with_fn']} ảnh có FN. Hai biểu hiện nhiều nhất theo số ảnh là {dominant_text}. Các nhóm có thể trùng nhau: output không parse được được scorer coi là không có dự đoán và có thể tạo FN. Có {error_summary['predictions_unknown_label']} box dùng lớp ngoài danh sách và {error_summary['boxes_outside_0_1000']} box vượt phạm vi tọa độ. FP chưa đủ để phân biệt box bịa với box đúng vật thể nhưng lệch vị trí; cần xem ảnh minh họa.

{chr(10).join(examples)}

### Câu 3 — Kiểm chứng thang tọa độ

Giữ nguyên output E1, chấm đúng norm1000 có F1={fnum(evidence['e1']['micro']['f1'])}; cố ý diễn giải cùng output là pixel có F1={fnum(wrong['micro']['f1'])}. Kết quả kiểm chứng lưu ở `report/coordinate_diagnostic.json`. Cùng một số x=500 tương ứng 320 pixel trên ảnh rộng 640 khi dùng thang 1000; diễn giải thành 500 pixel làm lệch box và giảm IoU. Công thức ngược là x_pixel=x/1000*width, y_pixel=y/1000*height. Nhãn được đổi từ YOLO tâm/kích thước sang hai góc, làm tròn và kẹp [0,1000]; đã kiểm tra ví dụ [0.5,0.5,0.2,0.2] thành [400,400,600,600] và vẽ lại sáu ảnh nhãn.

### Câu 4 — Mask loss

Chỉ phần assistant được giám sát. Prompt, token ảnh và padding có label=-100. Kiểm tra tiền tố token của prompt khớp chính xác chuỗi đầy đủ; decode phần không mask ra JSON và token kết thúc. Một ảnh có {model['image_tokens']} token ảnh; mẫu decode được lưu ở `report/model_checks.json`. Nếu tính loss trên cả chuỗi, nhiều token prompt/ảnh cố định sẽ làm loãng mục tiêu học vị trí và lớp trong đáp án, đồng thời buộc model dự đoán các token đầu vào mà nhiệm vụ không yêu cầu. Không cắt/truncate chuỗi làm mất token kết thúc đáp án.

### Câu 5 — Lượng tử hóa và projector float32

QLoRA giữ base LLM ở NF4 4-bit và học ma trận LoRA. Các trọng số lượng tử hóa không phải tham số float thông thường để cập nhật toàn phần bằng AdamW; vì vậy cấu hình skip `model.visual`, `visual` và `lm_head`. Với Transformers 5.19.0, quy tắc khớp prefix khiến tên ngắn `visual` chưa đủ để loại trừ đường dẫn `model.visual`; mã đã kiểm tra nhánh visual không chứa Linear4bit. Ở E2/E3, projector được giữ ở float32; mọi tham số LoRA trainable cũng được chuyển sang float32 để GradScaler không gặp gradient fp16. Matmul vẫn đi qua autocast. Float32 là lựa chọn ổn định số của lab, không phải quy luật rằng mọi projector chỉ có thể train ở float32. Checkpoint lưu adapter và `projector.pt` riêng; inference kiểm tra và nạp lại cả hai cho E2/E3.

### Câu 6 — Chia theo phiên quay

Frame cùng phiên gần nhau có thể rất giống. Chia ảnh ngẫu nhiên đưa các frame gần trùng vào train và test khiến F1 đánh giá quá lạc quan khả năng tổng quát hóa. Giữ nguyên split do công cụ đề tạo, kiểm tra không có session giao nhau giữa mọi cặp split. Số liệu thực tế:

{counts}

Phân bố lớp giữa các phiên không đều, đặc biệt val chỉ có {meta['counts']['val']['boxes']['pothole']} box pothole, nên cần đọc F1 theo từng lớp.

### Câu 7 — So sánh E1/E2/E3 và tài nguyên

{comparison}

Setup cơ sở E4 là {best}, chọn bằng micro-F1 trên {selection['validation_images']} ảnh validation cố định, không chọn bằng điểm test. Các điểm validation lưu trong `report/selection.json`. VRAM đỉnh đo được lớn nhất là {peak_key.upper()}; mở projector thêm gradient và trạng thái optimizer, còn E3 phải backpropagate qua ViT để học LoRA. Không thể suy ra lợi ích chỉ từ số tham số: phải đối chiếu F1, FP/ảnh, F1 từng lớp và chi phí thực tế trong bảng. CI95 của từng setup là khoảng cho từng điểm số; khoảng chồng lấn không tự chứng minh hai setup tương đương, và khoảng không chồng lấn cũng không thay thế kiểm định chênh lệch ghép cặp.

### Câu 8 — Khoảng tin cậy và giới hạn kết luận

Độ rộng CI95 F1: {widths}. Công cụ chấm bootstrap 1000 lần theo ảnh. Test có {len(entries)} ảnh, ít mẫu cho một số lớp và các ảnh trong cùng phiên có tương quan; bootstrap theo ảnh có thể đánh giá thấp bất định giữa phiên. Vì vậy các chênh lệch nhỏ chỉ là tín hiệu trên split này, chưa chứng minh ưu thế ổn định trên tuyến đường hoặc phiên quay khác. Không tối ưu prompt hay chọn checkpoint bằng test.

### Câu 9 — Phân tích theo lớp và hình minh họa

Trong {best}, lớp thấp nhất là {worst}, F1={fnum(evidence[best.lower()]['per_class'][worst]['f1'])}. Diện tích box trung vị trên test theo phần trăm ảnh: {area_text}. Box nhỏ nhạy với lệch vài pixel; số mẫu và sự giống nhau về hình dạng có thể góp phần gây lỗi. Đây là các giả thuyết, không phải kết luận nhân quả từ một lần chạy. Sáu ví dụ sau hiển thị box thật và dự đoán, kèm TP/FP/FN theo chính quy tắc ghép của scorer.

{chr(10).join(illustrations)}

### Câu 10 — Ablation E4

Thay duy nhất tỷ lệ ảnh train: 0.1 và 1.0, giữ setup {best}, rank, alpha, learning rate, batch, gradient accumulation, seed và số optimizer steps. Tập 10% là mẫu cố định theo seed 42; giá trị 100% dùng lại run cơ sở đúng cấu hình và chấm dưới tên E4. Cả hai đều 300 steps nên 10% được lặp lại nhiều hơn; phép thử đánh giá lượng dữ liệu độc nhất dưới cùng ngân sách cập nhật, không phải cùng số epoch.

{e4_description}

Diễn giải kết quả với CI95 và các giới hạn của test nhỏ ở câu 8; không khẳng định ý nghĩa thống kê chỉ từ hai khoảng riêng lẻ.

## Nhìn lại

Mã triển khai có kiểm tra prefix chat template, mask đáp án, module LoRA, loss/gradient hữu hạn và nạp lại projector. Scorer có thể parse JSON hợp lệ nhưng chưa chứng minh box đúng; cần kết hợp F1, FP/FN và ảnh. Chưa thực hiện E5 GRPO tùy chọn. Các kết quả phản ánh những lần chạy được ghi trong evidence và train logs, không thay thế việc tự xem lại mã và giải thích trong phần vấn đáp.
'''
    Path("REPORT.md").write_text(report, encoding="utf-8")
    print("REPORT.md generated from measured evidence; six prediction figures saved.")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--name", default="Nguyễn Văn Tứ")
    generate(parser.parse_args().name)


In [ ]:
%%writefile run_lab.py
"""Run the required experiments, score with original tools, and build a report.

Every model run is isolated in a new Python process to release CUDA memory.
Complete checkpoints and prediction files can be reused after an interruption.
"""
import argparse
import json
import os
import subprocess
import sys
import time
import zipfile
from pathlib import Path

from scripts.labcommon import read_jsonl
from scripts.score import evaluate


def load(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def run(command, log_path):
    print("\nRUN:", " ".join(map(str, command)), flush=True)
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    with Path(log_path).open("a", encoding="utf-8") as log:
        proc = subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
        returncode = proc.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


def complete_predictions(path, split, limit=None):
    path = Path(path)
    if not path.exists():
        return False
    rows = read_jsonl(f"data/splits/{split}.jsonl")
    if limit:
        rows = rows[:limit]
    try:
        predictions = read_jsonl(path)
    except (ValueError, KeyError):
        return False
    ids = [r["image_id"] for r in predictions]
    return len(ids) == len(set(ids)) and set(ids) == {r["image_id"] for r in rows}


def package_submission():
    bundle = Path("work/lab_submission.zip")
    bundle.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
        for item in ("student_lab.py", "run_lab.py", "make_report.py", "lab_day10.ipynb",
                     "build_notebook.py", "student_checks.py", "colab_runner.py", "data/splits/meta.json",
                     "REPORT.md", "README.md", "GUIDE.md", "RUBRIC.md", "requirements-colab.txt",
                     "pytest.ini", ".gitignore", ".env.example"):
            if Path(item).exists():
                archive.write(item, item)
        for directory in ("report", "configs", "scripts", "tests"):
            for path in Path(directory).rglob("*"):
                if path.is_file() and "__pycache__" not in path.parts:
                    archive.write(path, path.as_posix())
        for path in list(Path("work").glob("*.jsonl")) + list(Path("work").glob("*.run.json")):
            archive.write(path, path.as_posix())
    return bundle


def main(args):
    python = sys.executable
    root = Path(args.checkpoints)
    root.mkdir(parents=True, exist_ok=True)
    status_path = Path("work/pipeline_status.json")
    started = time.time()

    def status(stage, **extra):
        write(status_path, {"stage": stage, "started_unix": started,
                           "updated_unix": time.time(), **extra})

    def infer(name, checkpoint=None, split="test", limit=None):
        tag = name.lower() + ("_val" if split == "val" else "")
        out = f"work/{tag}.jsonl"
        if not complete_predictions(out, split, limit):
            status(f"infer {name} {split}")
            command = [python, "-u", "student_lab.py", "infer", "--name", name,
                       "--split", split, "--output", out, "--resume", "--batch-size", args.infer_batch]
            if checkpoint:
                command += ["--checkpoint", checkpoint]
            if limit:
                command += ["--limit", limit]
            run(command, f"work/logs/{tag}_infer.log")
        return out

    def score(name, pred, run_info=None, filename=None):
        command = [python, "scripts/score.py", "--pred", pred, "--name", name,
                   "--out", filename or f"report/evidence/{name.lower()}.json"]
        if run_info:
            for key in ("trainable_params", "peak_vram_gb", "train_minutes"):
                command += ["--meta", f"{key}={run_info[key]}"]
        run(command, f"work/logs/{name.lower()}_score.log")

    def train(name, setup, fraction=1.0):
        checkpoint = root / name.lower()
        run_path = checkpoint / "run.json"
        current = load(run_path) if run_path.exists() else None
        if current and (current["setup"] != setup or current["train_fraction"] != fraction):
            raise ValueError("Existing checkpoint has a different experiment configuration")
        if not current or current["completed_steps"] < current["max_steps"]:
            status(f"train {name}")
            command = [python, "-u", "student_lab.py", "train", "--name", name,
                       "--setup", setup, "--output", checkpoint, "--fraction", fraction,
                       "--batch-size", args.train_batch, "--grad-accum", args.grad_accum]
            if current and (checkpoint / "training_state.pt").exists():
                command += ["--resume", checkpoint]
            run(command, f"work/logs/{name.lower()}_train.log")
        current = load(run_path)
        # Keep compact run metadata/logs in the repository, but no weights.
        write(f"report/runs/{name.lower()}.json", current)
        source_log = checkpoint / "train_log.jsonl"
        Path(f"report/runs/{name.lower()}_train.jsonl").write_text(source_log.read_text(), encoding="utf-8")
        return checkpoint, current

    try:
        status("check data and model")
        import importlib.metadata
        environment = {"python": sys.version}
        for package in ("torch", "torchvision", "transformers", "peft", "accelerate", "bitsandbytes", "torchao"):
            try:
                environment[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                environment[package] = None
        write("report/environment.json", environment)
        run([python, "student_lab.py", "inspect-data"], "work/logs/checks.log")
        run([python, "student_lab.py", "verify-model"], "work/logs/checks.log")
        e0_predictions = infer("E0")
        e0_run = load(Path(e0_predictions).with_suffix(".run.json"))
        score("E0", e0_predictions, {"trainable_params": 0, "train_minutes": 0,
                                     "peak_vram_gb": e0_run["peak_vram_gb"]})
        validation = {}
        for setup in ("E1", "E2", "E3"):
            checkpoint, info = train(setup, setup)
            score(setup, infer(setup, checkpoint), info)
            val_pred = infer(setup, checkpoint, "val", args.val_images)
            val_rows = read_jsonl("data/splits/val.jsonl")[:args.val_images]
            raw = {r["image_id"]: r["raw"] for r in read_jsonl(val_pred)}
            result = evaluate(val_rows, raw)
            result["split"] = "val_subset"
            result["n_selected"] = len(val_rows)
            write(f"report/validation/{setup.lower()}.json", result)
            validation[setup] = result["micro"]["f1"]
        best = max(validation, key=validation.get)
        write("report/selection.json", {"best_setup": best, "validation_images": args.val_images,
                                        "selection_metric": "micro_f1", "validation_scores": validation})
        print("Selected E4 baseline from validation:", best, validation, flush=True)
        checkpoint, info = train("E4_data_size_0.1", best, 0.1)
        score("E4_data_size_0.1", infer("E4_data_size_0.1", checkpoint), info)
        info = load(root / best.lower() / "run.json")
        score("E4_data_size_1.0", f"work/{best.lower()}.jsonl", info)
        status("build report")
        run([python, "make_report.py", "--name", args.name], "work/logs/report.log")
        status("complete")
        bundle = package_submission()
        print(f"All mandatory experiments completed. Artifact: {bundle.resolve()}", flush=True)
    except Exception as exc:
        status("failed", error=str(exc))
        raise


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoints", default="work/checkpoints")
    parser.add_argument("--train-batch", type=int, default=2)
    parser.add_argument("--grad-accum", type=int, default=4)
    parser.add_argument("--infer-batch", type=int, default=4)
    parser.add_argument("--val-images", type=int, default=40)
    parser.add_argument("--name", default="Nguyễn Văn Tứ")
    main(parser.parse_args())


In [ ]:
%%writefile colab_runner.py
"""Run inside a Colab cell; download recoverable checkpoints and measured artifacts."""
import json
import subprocess
import sys
import threading
import zipfile
from pathlib import Path


def backup_checkpoint(message):
    directory = Path(message["directory"])
    bundle = Path("work/backups") / f"day10_{message['name'].lower()}_step{message['step']}.zip"
    bundle.parent.mkdir(parents=True, exist_ok=True)
    temporary = bundle.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for path in directory.rglob("*"):
            if path.is_file():
                archive.write(path, (Path("work/checkpoints") / directory.name / path.relative_to(directory)).as_posix())
    temporary.replace(bundle)
    return bundle


def run_colab(extra_args=(), auto_download=True):
    from google.colab import files
    from run_lab import package_submission
    command = [sys.executable, "-u", "run_lab.py", *map(str, extra_args)]
    Path("work/logs").mkdir(parents=True, exist_ok=True)
    with Path("work/logs/colab_supervisor.log").open("a", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, start_new_session=True)
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
            if line.startswith("CHECKPOINT_READY "):
                message = json.loads(line.removeprefix("CHECKPOINT_READY "))
                bundle = backup_checkpoint(message)
                print(f"BACKUP_DOWNLOAD {bundle} ({bundle.stat().st_size / 1024**2:.1f} MiB)", flush=True)
                if auto_download:
                    files.download(str(bundle))
            elif "[test] n=" in line:
                bundle = package_submission()
                print("ARTIFACT_DOWNLOAD", bundle, flush=True)
                if auto_download:
                    files.download(str(bundle))
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    if auto_download:
        files.download(str(package_submission()))
    print("PIPELINE_COMPLETE", flush=True)


def start_colab(extra_args=()):
    """Release the notebook kernel so file download comms can run during training."""
    thread = threading.Thread(target=run_colab, args=(extra_args, False), daemon=True,
                              name="day10-lab-supervisor")
    thread.start()
    return thread


_requested_downloads = set()


def poll_colab():
    """Run in a short foreground cell so Colab can process file download messages."""
    from google.colab import files
    status = json.loads(Path("work/pipeline_status.json").read_text())
    print(json.dumps(status, ensure_ascii=False, indent=2))
    log = Path("work/logs/colab_supervisor.log")
    if log.exists():
        print(log.read_text(encoding="utf-8")[-1200:])
    bundles = sorted(Path("work/backups").glob("day10_*.zip"))
    artifact = Path("work/lab_submission.zip")
    if artifact.exists():
        bundles.append(artifact)
    for bundle in bundles:
        stamp = bundle.stat()
        key = (str(bundle), stamp.st_size, stamp.st_mtime_ns)
        if key not in _requested_downloads:
            files.download(str(bundle))
            _requested_downloads.add(key)
            print("DOWNLOAD_REQUESTED", bundle, stamp.st_size, flush=True)
    return status


## 7. Suy luận và chấm E1–E3
Nạp base model mới, adapter và projector tương ứng; chạy test đủ ảnh, ghi raw output, chấm bằng scripts/score.py với số tham số, VRAM và thời gian thực tế. 40 ảnh đầu của val cố định dùng chọn setup cơ sở E4; báo cáo ghi rõ đây là subset.

In [ ]:
CHECKPOINT_ROOT = 'work/checkpoints'
print('Checkpoint directory:', CHECKPOINT_ROOT)


## 8. E4: ablation dữ liệu và chạy toàn bộ pipeline
Chọn cơ sở bằng validation F1, so sánh 10% và 100% ảnh train, giữ các siêu tham số khác như nhau. Run 100% được dùng lại; run 10% bắt đầu từ model gốc. E5 GRPO là phần tùy chọn chưa triển khai. Nếu dừng giữa chừng, chạy lại pipeline để tiếp tục từ checkpoint và các dự đoán đã ghi.

In [ ]:
from colab_runner import start_colab
if 'lab_thread' in globals() and lab_thread.is_alive():
    raise RuntimeError('Pipeline đang chạy; dùng ô theo dõi bên dưới.')
lab_thread = start_colab(['--checkpoints', CHECKPOINT_ROOT])
print('Pipeline đang chạy nền. Chờ stage=complete ở ô kiểm tra tiếp theo.')


In [ ]:
from colab_runner import poll_colab
state = poll_colab()
print('Chạy lại ô này để xem tiến độ và tải checkpoint mới.')


## 9. Báo cáo và xuất bài
REPORT được tạo sau khi đủ evidence. Sáu ảnh có GT xanh và prediction đỏ; toàn bộ nhận xét lấy số từ các run thật. Check_submission cần ít nhất 5 commit trong repo nộp; commit từ các giai đoạn phát triển trên máy cá nhân và kiểm tra lại trước khi đẩy GitHub.

In [ ]:
assert json.loads(Path('work/pipeline_status.json').read_text())['stage'] == 'complete', 'Chờ pipeline hoàn thành'
from IPython.display import Markdown
display(Markdown(Path('REPORT.md').read_text()))
!python scripts/check_submission.py


In [ ]:
from google.colab import files
files.download('work/lab_submission.zip')
